# IPO-Ready: Python financial analytics baseline

Retrospective Aether RHP pilot. Curated transcription, not automatic extraction. Monetary inputs are INR million; EPS is INR/share. Run from the project root or notebooks folder. No model or investment recommendation.


In [ ]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd() if (Path.cwd() / 'research').exists() else Path.cwd().parent
pilot = json.loads((root / 'research/aether_reconciliation/pilot_data.json').read_text())
periods = ['9M Dec-2021', '9M Dec-2020', 'FY21', 'FY20', 'FY19']
rows = []
for row in pilot['raw']:
    for i, period in enumerate(periods):
        rows.append({'metric': row[0], 'period': period, 'value': row[i+1], 'unit': row[6], 'pdf_page': row[7], 'basis': row[9]})
facts = pd.DataFrame(rows)
assert len(facts) == 110
facts.head()


## Prepare comparable annual observations

Keep interim periods separate. Revenue and cash figures below are converted from million to crore by dividing by ten. EPS must not receive that conversion.


In [ ]:
annual = facts[facts.period.isin(['FY19', 'FY20', 'FY21'])].pivot(index='period', columns='metric', values='value').reindex(['FY19','FY20','FY21'])
profile = pd.DataFrame(index=annual.index)
profile['Revenue (INR crore)'] = annual['Revenue from operations'] / 10
profile['PAT (INR crore)'] = annual['Profit after tax'] / 10
profile['CFO (INR crore)'] = annual['Net cash from operating activities'] / 10
profile['EBITDA margin'] = annual['EBITDA excluding other income'] / annual['Revenue from operations']
profile['CFO/PAT'] = annual['Net cash from operating activities'] / annual['Profit after tax']
profile['Borrowings/equity'] = annual['Debt as disclosed'] / annual['Total equity']
profile


## Growth and reconciliation

Use two elapsed years from FY19 to FY21. These internal checks do not establish financial health or validate transcription against the original PDF.


In [ ]:
revenue_cagr = (annual.loc['FY21','Revenue from operations'] / annual.loc['FY19','Revenue from operations']) ** (1/2) - 1
residual = annual['Revenue from operations'] + annual['Other income'] - annual['Total income']
assert residual.abs().max() <= 0.02
assert abs(profile.loc['FY21', 'CFO/PAT'] - 231.96/711.19) < 1e-12
print(f'Revenue CAGR FY19-FY21: {revenue_cagr:.2%}')
print(profile.round(3).to_string())
output = root / 'outputs/poc'
output.mkdir(parents=True, exist_ok=True)
profile.to_csv(output / 'aether_annual_profile.csv')


## Interpretation and unresolved issues

Revenue and PAT growth should be considered alongside cash conversion. CFO/PAT is descriptive, not a distress probability. The pilot retains conflicting repeated Dec-2020 asset values and unresolved valuation inputs. Source: https://www.sebi.gov.in/sebi_data/attachdocs/may-2022/1652869529567.pdf (physical pp.64-67, 302-303).

The web demo adds evidence drill-down and an executed deterministic workflow trace. Model-selected actions and automatic extraction are future milestones.
